# CUAD Data Inspection and Cleaning

This notebook continues from the extracted and split CUAD dataset. 
The goal is to inspect the training and testing data, identify data quality issues, and prepare clean data for later preprocessing and modeling.

In [12]:
import json
import pandas as pd

### Loading the Training and Testing Data

I will first load the training and testing JSON files created during the previous data extraction and splitting step. These datasets will be inspected for missing values, duplicates, formatting issues, and other data quality problems before preprocessing.

In [13]:
with open("../data/train_separate_questions.json", "r") as file:
    train_json = json.load(file)

with open("../data/test.json", "r") as file:
    test_json = json.load(file)

### Inspecting the Data

Before cleaning the datasets, I will inspect their structure and contents to better understand the data I am working with.

In [14]:
type(train_json), type(test_json)

(dict, dict)

In [15]:
train_json.keys(), test_json.keys()

(dict_keys(['version', 'data']), dict_keys(['version', 'data']))

In [16]:
type(train_json["data"]), type(test_json["data"])

(list, list)

In [24]:
len(train_json["data"]),len(test_json["data"])

(408, 102)

In [25]:
train_json["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [26]:
train_json["data"][0]["title"]

'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT'

In [27]:
type(train_json["data"][0]["paragraphs"]),len(train_json["data"][0]["paragraphs"])

(list, 1)

In [28]:
train_json["data"][0]["paragraphs"][0].keys()

dict_keys(['qas', 'context'])

In [29]:
context = train_json["data"][0]["paragraphs"][0]["context"]

type(context),len(context)

(str, 54290)

In [30]:
qas = train_json["data"][0]["paragraphs"][0]["qas"]

type(qas),len(qas)

(list, 68)

In [31]:
qas[0].keys()

dict_keys(['answers', 'id', 'question', 'is_impossible'])

In [32]:
qas[0]

{'answers': [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}],
 'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Document Name_0',
 'question': 'Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract',
 'is_impossible': False}

In [33]:
qas[0]["answers"]

[{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}]

In [34]:
impossible_qa = next(qa for qa in qas if qa["is_impossible"])

impossible_qa

{'answers': [],
 'id': 'LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT__Notice Period To Terminate Renewal_0',
 'question': 'Highlight the parts (if any) of this contract related to "Notice Period To Terminate Renewal" that should be reviewed by a lawyer. Details: What is the notice period required to terminate renewal?',
 'is_impossible': True}

### Understanding the QA Annotations

Each QA entry contains an `is_impossible` field and an `answers` list. If the clause is present in the contract, `is_impossible` is `False` and the answers list contains the annotated text and its starting position. If the clause is not present, `is_impossible` is `True` and the answers list is empty.

## Checking Data Quality

### Checking for Missing or Empty Fields

I will check whether important fields in the training and testing datasets are missing or empty before making any changes to the data.

In [35]:
empty_titles = 0
empty_paragraphs = 0
empty_contexts = 0

for contract in train_json["data"]:
    if not contract.get("title"): empty_titles += 1
    if not contract.get("paragraphs"): empty_paragraphs += 1
    for paragraph in contract.get("paragraphs", []):
        if not paragraph.get("context"): empty_contexts += 1

print("Empty titles:", empty_titles)
print("Empty paragraphs:", empty_paragraphs)
print("Empty contexts:", empty_contexts)

Empty titles: 0
Empty paragraphs: 0
Empty contexts: 0


#### Checking QA Fields

Next, I will check the QA entries for missing questions, IDs, and `is_impossible` values.

In [36]:
empty_questions = 0
empty_ids = 0
missing_impossible = 0

for contract in train_json["data"]:
    for paragraph in contract["paragraphs"]:
        for qa in paragraph["qas"]:
            if not qa.get("question"): empty_questions += 1
            if not qa.get("id"): empty_ids += 1
            if "is_impossible" not in qa: missing_impossible += 1

print("Empty questions:", empty_questions)
print("Empty IDs:", empty_ids)
print("Missing is_impossible:", missing_impossible)

Empty questions: 0
Empty IDs: 0
Missing is_impossible: 0


### Checking Annotation Consistency
I will check whether the `is_impossible` values are consistent with the answers provided for each QA entry.

In [37]:
inconsistent = 0

for contract in train_json["data"]:
    for paragraph in contract["paragraphs"]:
        for qa in paragraph["qas"]:
            if qa["is_impossible"] and len(qa["answers"]) > 0:
                inconsistent += 1
            if not qa["is_impossible"] and len(qa["answers"]) == 0:
                inconsistent += 1

print("Inconsistent annotations:", inconsistent)

Inconsistent annotations: 0


### Checking for Duplicate QA IDs

I will check whether any QA entries have duplicate IDs in the training dataset.

In [38]:
ids = []
for contract in train_json["data"]:
    for paragraph in contract["paragraphs"]:
        for qa in paragraph["qas"]:
            ids.append(qa["id"])
duplicates = len(ids) - len(set(ids))
print("Duplicate QA IDs:", duplicates)

Duplicate QA IDs: 0


### Checking the Testing Dataset
I will perform the same data quality checks on the testing dataset to make sure it does not contain missing fields or inconsistent annotations.

In [39]:
empty_titles = 0
empty_paragraphs = 0
empty_contexts = 0

for contract in test_json["data"]:
    if not contract.get("title"): empty_titles += 1
    if not contract.get("paragraphs"): empty_paragraphs += 1
    for paragraph in contract.get("paragraphs", []):
        if not paragraph.get("context"): empty_contexts += 1

print("Empty titles:", empty_titles)
print("Empty paragraphs:", empty_paragraphs)
print("Empty contexts:", empty_contexts)

Empty titles: 0
Empty paragraphs: 0
Empty contexts: 0


In [40]:
empty_questions = 0
empty_ids = 0
missing_impossible = 0

for contract in test_json["data"]:
    for paragraph in contract["paragraphs"]:
        for qa in paragraph["qas"]:
            if not qa.get("question"): empty_questions += 1
            if not qa.get("id"): empty_ids += 1
            if "is_impossible" not in qa: missing_impossible += 1

print("Empty questions:", empty_questions)
print("Empty IDs:", empty_ids)
print("Missing is_impossible:", missing_impossible)

Empty questions: 0
Empty IDs: 0
Missing is_impossible: 0


In [41]:
inconsistent = 0

for contract in test_json["data"]:
    for paragraph in contract["paragraphs"]:
        for qa in paragraph["qas"]:
            if qa["is_impossible"] and len(qa["answers"]) > 0:
                inconsistent += 1
            if not qa["is_impossible"] and len(qa["answers"]) == 0:
                inconsistent += 1

print("Inconsistent annotations:", inconsistent)

Inconsistent annotations: 0


In [42]:
ids = []
for contract in test_json["data"]:
    for paragraph in contract["paragraphs"]:
        for qa in paragraph["qas"]:
            ids.append(qa["id"])
duplicates = len(ids) - len(set(ids))
print("Duplicate QA IDs:", duplicates)

Duplicate QA IDs: 0


### Checking Answer Positions
I will check whether each annotated answer matches the contract text at its specified `answer_start` position.

In [43]:
wrong_positions = 0

for contract in train_json["data"]:
    for paragraph in contract["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            for answer in qa["answers"]:
                start = answer["answer_start"]
                text = answer["text"]
                if context[start:start + len(text)] != text:
                    wrong_positions += 1
print("Wrong answer positions:", wrong_positions)

Wrong answer positions: 0


In [44]:
wrong_positions = 0

for contract in test_json["data"]:
    for paragraph in contract["paragraphs"]:
        context = paragraph["context"]
        for qa in paragraph["qas"]:
            for answer in qa["answers"]:
                start = answer["answer_start"]
                text = answer["text"]
                if context[start:start + len(text)] != text:
                    wrong_positions += 1

print("Wrong answer positions:", wrong_positions)

Wrong answer positions: 0


### Checking for Duplicate Contracts
I will check whether any contract titles are duplicated within the training or testing datasets.

In [45]:
train_titles = []
test_titles = []

for contract in train_json["data"]:
    train_titles.append(contract["title"])
for contract in test_json["data"]:
    test_titles.append(contract["title"])
print("Duplicate training contracts:", len(train_titles) - len(set(train_titles)))
print("Duplicate testing contracts:", len(test_titles) - len(set(test_titles)))

Duplicate training contracts: 0
Duplicate testing contracts: 0


### Checking for Train-Test Overlap
I will check whether any contracts appear in both the training and testing datasets.

In [46]:
overlap = set(train_titles) & set(test_titles)
print("Contracts in both train and test:", len(overlap))

Contracts in both train and test: 0


## Data Cleaning Summary
The training and testing datasets were checked for missing or empty fields, inconsistent QA annotations, duplicate QA IDs, incorrect answer positions, duplicate contracts, and overlap between the training and testing sets. No issues were found in these checks. The datasets are consistent and ready for the next stage of the project.